# 00 — Preparação de Dados para Modelagem de Probabilidade de Churn
## PRT Seguros

Este notebook prepara os dados que serão usados por **todos** os notebooks de modelagem
(`01`–`05`). O objetivo final não é apenas classificar 0/1, e sim estimar a
**probabilidade contínua entre 0 e 1** de cada indivíduo dar churn.

**Base de treino:** `Base_Unificada_Outer.csv` (tem a coluna `churned`)
**Base de teste/Kaggle:** `Base_Unificada_Kaggle_Outer.csv` (não tem `churned` — é o que vamos prever)

**Fluxo:**
1. Carregar as duas bases
2. Alinhar nomes de colunas (a base Kaggle usa capitalização diferente em algumas dummies)
3. Corrigir um bug de encoding de região e remover leakage/datas/colunas que não existem nas duas bases
4. Fazer o split treino/validação **antes** de ajustar qualquer transformação (evita vazamento)
5. Reproduzir a clusterização K-Means (`clusterização/clusterizacao_clientes.ipynb`) — ajustada
   **somente na partição de treino** e aplicada em validação e Kaggle, gerando a feature `cluster`
6. Imputar nulos (mediana calculada apenas na partição de treino)
7. Salvar `train_model_ready.csv`, `val_model_ready.csv` e `kaggle_model_ready.csv` em `dados_processados/`

> **Nota sobre a versão anterior deste notebook:** o K-Means, o `StandardScaler` e o `SimpleImputer`
> estavam sendo ajustados em 100% da base de treino, e só depois cada notebook de modelo fazia seu
> próprio `train_test_split`. Isso deixava a validação levemente "em casa" — o K-Means já tinha visto
> aqueles pontos ao decidir os centróides. Agora o split acontece primeiro, e todo o resto é ajustado
> **apenas na partição de treino**.


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## 2. Carregar as bases

> **Observação importante:** a coluna `churned` (0 = ficou, 1 = deu churn) só existe na base de treino.
> A base Kaggle é a que vamos usar como teste final — ela não tem rótulo, por isso a métrica de
> validação (AUC-ROC) sempre será medida na partição de validação, nunca na base Kaggle.

In [2]:
df_train = pd.read_csv("../bases/tabelas_unificadas/Base_Unificada_Outer.csv")
df_kaggle = pd.read_csv("../bases/bases_kaggle/Base_Unificada_Kaggle_Outer.csv")

print(f"Treino (Base_Unificada_Outer)        : {df_train.shape}")
print(f"Kaggle (Base_Unificada_Kaggle_Outer) : {df_kaggle.shape}")
print(f"Taxa de churn no treino              : {df_train['churned'].mean():.2%}")


Treino (Base_Unificada_Outer)        : (100000, 84)
Kaggle (Base_Unificada_Kaggle_Outer) : (100000, 77)
Taxa de churn no treino              : 12.09%


## 3. Alinhar nomes de colunas entre treino e Kaggle

As duas bases foram tratadas por pipelines diferentes e ficaram com capitalização distinta em
algumas colunas dummy (ex.: `escolaridade_fundamental` no Kaggle vs. `escolaridade_Fundamental` no treino).
Sem esse ajuste, o modelo treinado nunca reconheceria essas colunas na hora de prever no Kaggle.

In [3]:
train_cols_lower = {c.lower(): c for c in df_train.columns}
rename_map = {
    c: train_cols_lower[c.lower()]
    for c in df_kaggle.columns
    if c not in df_train.columns and c.lower() in train_cols_lower
}
print(f"Colunas renomeadas no Kaggle para bater com o treino ({len(rename_map)}):")
for k, v in rename_map.items():
    print(f"  {k:<35} -> {v}")

df_kaggle = df_kaggle.rename(columns=rename_map)


Colunas renomeadas no Kaggle para bater com o treino (8):
  escolaridade_fundamental            -> escolaridade_Fundamental
  escolaridade_medio                  -> escolaridade_Medio
  escolaridade_pos                    -> escolaridade_Pos
  escolaridade_superior               -> escolaridade_Superior
  canal_aquisicao_agente              -> canal_aquisicao_Agente
  canal_aquisicao_digital             -> canal_aquisicao_Digital
  canal_aquisicao_indicacao           -> canal_aquisicao_Indicacao
  canal_aquisicao_telefone            -> canal_aquisicao_Telefone


## 4. Corrigir bug de encoding em `regiao_*`

A base de treino tem **4 colunas de região que representam a mesma coisa** por inconsistência na
limpeza de dados: `regiao_Centro`, `regiao_Centro-Oeste`, `regiao_Oeste` e `regiao_Regiao Oeste`
somam ~33% dos clientes, mas só `regiao_Centro-Oeste` existe na base Kaggle (lá, já vem normalizada
para essa única categoria).

Sem esse merge, ao dropar as 3 colunas extras (próximo passo), **28% das linhas de treino perderiam
toda a informação de região** (ficariam com todas as colunas de região zeradas), enquanto no Kaggle
nenhum cliente perde essa informação — um caso claro de *distribution shift* introduzido pelo próprio
pipeline, não pelos dados originais.

In [4]:
REGIAO_CENTRO_OESTE_VARIANTES = ["regiao_Centro", "regiao_Oeste", "regiao_Regiao Oeste"]

antes = df_train["regiao_Centro-Oeste"].sum()
df_train["regiao_Centro-Oeste"] = df_train[
    ["regiao_Centro-Oeste"] + REGIAO_CENTRO_OESTE_VARIANTES
].max(axis=1)
depois = df_train["regiao_Centro-Oeste"].sum()

print(f"regiao_Centro-Oeste antes do merge : {int(antes):,} clientes ({antes/len(df_train):.1%})")
print(f"regiao_Centro-Oeste depois do merge: {int(depois):,} clientes ({depois/len(df_train):.1%})")
print(f"regiao_Centro-Oeste no Kaggle       : {int(df_kaggle['regiao_Centro-Oeste'].sum()):,} clientes "
      f"({df_kaggle['regiao_Centro-Oeste'].mean():.1%})")


regiao_Centro-Oeste antes do merge : 10,126 clientes (10.1%)
regiao_Centro-Oeste depois do merge: 33,553 clientes (33.6%)
regiao_Centro-Oeste no Kaggle       : 33,140 clientes (34.9%)


## 5. Remover leakage, datas e colunas que não existem nas duas bases

- `score_propensao_churn`, `cluster_sugerido_crm`: métricas do CRM derivadas do próprio churn (leakage)
- `data_primeira_apolice`, `data_nascimento`: datas em string, já representadas por `tempo_cliente_dias` e `idade`
- `renovacoes_consecutivas`: correlação de 0.94 com `tempo_cliente_dias` (multicolinearidade)
- `num_apolices_basica`, `num_apolices_padrao`, `num_apolices_premium`: existem só no treino
- `regiao_Centro`, `regiao_Oeste`, `regiao_Regiao Oeste`: já incorporadas em `regiao_Centro-Oeste` no passo anterior

In [5]:
DROP_COLS = [
    "score_propensao_churn", "cluster_sugerido_crm",
    "data_primeira_apolice", "data_nascimento",
    "renovacoes_consecutivas",
    "num_apolices_basica", "num_apolices_padrao", "num_apolices_premium",
    "regiao_Centro", "regiao_Oeste", "regiao_Regiao Oeste",
]

df_train = df_train.drop(columns=[c for c in DROP_COLS if c in df_train.columns])
df_kaggle = df_kaggle.drop(columns=[c for c in DROP_COLS if c in df_kaggle.columns])

feat_train = set(df_train.columns) - {"churned"}
feat_kaggle = set(df_kaggle.columns)

assert feat_train == feat_kaggle, (
    f"Colunas ainda divergentes!\n"
    f"Só no treino: {sorted(feat_train - feat_kaggle)}\n"
    f"Só no kaggle: {sorted(feat_kaggle - feat_train)}"
)
print(f"OK — {len(feat_train)} colunas de feature idênticas em treino e Kaggle.")
print(f"Shape treino : {df_train.shape}")
print(f"Shape kaggle : {df_kaggle.shape}")


OK — 72 colunas de feature idênticas em treino e Kaggle.
Shape treino : (100000, 73)
Shape kaggle : (100000, 72)


## 6. Split treino/validação — feito ANTES de ajustar qualquer transformação

Esse é o ponto-chave para uma validação honesta: o K-Means, o `StandardScaler` e o `SimpleImputer`
só podem "ver" a partição de treino. A partição de validação e a base Kaggle são tratadas como dados
totalmente novos, sempre com `.transform()`/`.predict()`, nunca com `.fit()`.

Usamos o mesmo `test_size=0.2` e `random_state=42` que os notebooks de modelo usavam antes — assim o
split fica *fixo* e idêntico para todos os 5 modelos, o que também torna a comparação entre eles mais justa.

In [6]:
ID_COL = "cod_individuo"
TARGET_COL = "churned"

df_train_split, df_val_split = train_test_split(
    df_train, test_size=0.2, stratify=df_train[TARGET_COL], random_state=RANDOM_STATE
)
df_train_split = df_train_split.reset_index(drop=True)
df_val_split = df_val_split.reset_index(drop=True)

print(f"Treino     : {df_train_split.shape} | churn = {df_train_split[TARGET_COL].mean():.1%}")
print(f"Validação  : {df_val_split.shape} | churn = {df_val_split[TARGET_COL].mean():.1%}")


Treino     : (80000, 73) | churn = 12.1%
Validação  : (20000, 73) | churn = 12.1%


## 7. Clusterização K-Means — feature `cluster`

Reproduzimos a segmentação do notebook `clusterização/clusterizacao_clientes.ipynb`, ajustada **apenas
na partição de treino** e aplicada (só `transform`/`predict`) em validação e Kaggle.

As 3 features de contagem de apólices por tipo (`num_apolices_basica/padrao/premium`) foram removidas
da lista original de 26 features do notebook de clusterização, pois não existem na base Kaggle.

In [7]:
FEATURES_CLUSTER = [
    "tempo_cliente_dias", "num_apolices_ativas", "num_produtos_contratados",
    "desconto_aplicado_pct", "indice_relacionamento", "satisfacao_nps",
    "tipo_cobertura_basica", "estado_civil", "tipo_cobertura_premium",
    "tem_filhos", "renda_anual", "qtd_dependentes", "valor_imovel",
    "valor_cobertura_total", "idade", "score_engajamento_digital",
    "pagamento_em_dia", "num_reclamacoes_12m", "num_acessos_app_mes",
    "possui_imovel", "tipo_cobertura_padrao", "valor_premio_anual", "franquia_media",
]
K_FINAL = 7  # K=7 (ajustado apos teste sistematico de varios K - ver README)

imp_cluster = SimpleImputer(strategy="median").fit(df_train_split[FEATURES_CLUSTER])
scaler_cluster = StandardScaler().fit(imp_cluster.transform(df_train_split[FEATURES_CLUSTER]))

X_cluster_train = scaler_cluster.transform(imp_cluster.transform(df_train_split[FEATURES_CLUSTER]))
X_cluster_val = scaler_cluster.transform(imp_cluster.transform(df_val_split[FEATURES_CLUSTER]))
X_cluster_kaggle = scaler_cluster.transform(imp_cluster.transform(df_kaggle[FEATURES_CLUSTER]))

kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(X_cluster_train)

df_train_split["cluster"] = kmeans.predict(X_cluster_train)
df_val_split["cluster"] = kmeans.predict(X_cluster_val)
df_kaggle["cluster"] = kmeans.predict(X_cluster_kaggle)

churn_por_cluster = df_train_split.groupby("cluster")[TARGET_COL].agg(["mean", "count"])
churn_por_cluster.columns = ["taxa_churn", "n_clientes"]
print(churn_por_cluster.sort_values("taxa_churn", ascending=False))


         taxa_churn  n_clientes
cluster                        
0          0.221409       22944
6          0.196117         515
4          0.184543        4438
2          0.178893       12991
1          0.043986       15323
5          0.029181       10692
3          0.027869       13097


## 8. Imputar nulos restantes

Mediana calculada **apenas na partição de treino** e aplicada em validação e Kaggle.

In [8]:
feature_cols = [c for c in df_train_split.columns if c not in (ID_COL, TARGET_COL)]
print(f"Total de features finais: {len(feature_cols)}")

imputer_final = SimpleImputer(strategy="median").fit(df_train_split[feature_cols])

def montar_final(df_origem, tem_target):
    saida = pd.DataFrame(
        imputer_final.transform(df_origem[feature_cols]), columns=feature_cols
    )
    saida[ID_COL] = df_origem[ID_COL].values
    if tem_target:
        saida[TARGET_COL] = df_origem[TARGET_COL].values
    return saida

train_final = montar_final(df_train_split, tem_target=True)
val_final = montar_final(df_val_split, tem_target=True)
kaggle_final = montar_final(df_kaggle, tem_target=False)

assert train_final.isna().sum().sum() == 0
assert val_final.isna().sum().sum() == 0
assert kaggle_final.isna().sum().sum() == 0
print(f"train_final : {train_final.shape} — sem nulos")
print(f"val_final   : {val_final.shape} — sem nulos")
print(f"kaggle_final: {kaggle_final.shape} — sem nulos")


Total de features finais: 72


train_final : (80000, 74) — sem nulos
val_final   : (20000, 74) — sem nulos
kaggle_final: (100000, 73) — sem nulos


## 9. Salvar bases prontas para modelagem

Todos os notebooks de modelo (`01` a `05`) partem destes três arquivos — assim garantimos que
**todo modelo vê exatamente o mesmo treino, a mesma validação e o mesmo conjunto de features**,
sem nenhuma etapa de pré-processamento ter visto a validação ou o Kaggle antes da hora.

In [9]:
import os

os.makedirs("dados_processados", exist_ok=True)

train_final.to_csv("dados_processados/train_model_ready.csv", index=False)
val_final.to_csv("dados_processados/val_model_ready.csv", index=False)
kaggle_final.to_csv("dados_processados/kaggle_model_ready.csv", index=False)

print("Arquivos salvos em dados_processados/:")
print(f"  train_model_ready.csv  -> {train_final.shape}")
print(f"  val_model_ready.csv    -> {val_final.shape}")
print(f"  kaggle_model_ready.csv -> {kaggle_final.shape}")


Arquivos salvos em dados_processados/:
  train_model_ready.csv  -> (80000, 74)
  val_model_ready.csv    -> (20000, 74)
  kaggle_model_ready.csv -> (100000, 73)


## 10. Registrar esta preparação no MLflow (SageMaker)

Registramos as decisões desta etapa (shapes, features removidas, K do K-Means) como parâmetros de
uma run no MLflow — não é um modelo, é o "log" de como os dados de entrada dos notebooks `01`-`25`
foram construídos. O tracking server é o app **`equipe5`** do SageMaker Studio.

In [10]:
import os
import mlflow

MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI", "arn:aws:sagemaker:us-east-2:906513713169:mlflow-tracking-server/equipe5"
)
MLFLOW_EXPERIMENT = "churn-prt-seguros"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="00_preparacao_dados"):
    mlflow.log_params({
        "base_treino": "Base_Unificada_Outer.csv",
        "base_kaggle": "Base_Unificada_Kaggle_Outer.csv",
        "n_linhas_treino": len(df_train),
        "n_linhas_kaggle": len(df_kaggle),
        "n_features_finais": len(feature_cols),
        "n_features_cluster": len(FEATURES_CLUSTER),
        "k_means_k": K_FINAL,
        "colunas_removidas_leakage": DROP_COLS,
        "test_size_val": 0.2,
        "random_state": RANDOM_STATE,
    })
    mlflow.log_metric("taxa_churn_treino", float(df_train_split[TARGET_COL].mean()))
    mlflow.log_metric("taxa_churn_val", float(df_val_split[TARGET_COL].mean()))
    print(f"Run registrada no MLflow (experimento '{MLFLOW_EXPERIMENT}').")


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:13:01 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


Run registrada no MLflow (experimento 'churn-prt-seguros').
🏃 View run 00_preparacao_dados at: http://127.0.0.1:5000/#/experiments/1/runs/101817dbf496446fab82293722954787
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
